# 21 - Geracao de base interim pelos NCMs de supermercados

Este notebook cria uma variacao dos Parquets de `data/interim/items-notas-fiscais/`, mantendo somente itens cujo NCM completo aparece em `data/interim/ncms_supermercados.csv` (coluna `numero_ncm`). O CSV e resultado da analise do notebook 13.

O filtro usa igualdade entre codigos NCM normalizados para oito digitos, nao prefixos. Sao mantidas apenas linhas com descricao de produto/servico. Os arquivos mensais de origem nao sao alterados; as saidas sao gravadas em `data/interim/items-notas-fiscais-ncms-supermercados/`.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

# Configuracoes da analise e caminhos relativos ao repositorio.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_ROOT = PROJECT_ROOT / "data" / "interim"
SOURCE_ITEMS_DIR = INTERIM_ROOT / "2025"
NCM_CSV_PATH = INTERIM_ROOT / "ncms_supermercados_filtrados.csv"
OUTPUT_DIR = INTERIM_ROOT / "2025_NCMs_filtrados"

In [2]:
def find_col(columns, *patterns):
    """Return the first column whose normalized name contains every pattern."""
    normalized_columns = {column: re.sub(r"\s+", " ", str(column)).upper() for column in columns}
    for column, normalized in normalized_columns.items():
        if all(pattern.upper() in normalized for pattern in patterns):
            return column
    raise KeyError(f"Nenhuma coluna encontrada para os padroes: {patterns}")


def normalize_ncm(series):
    """Normalize NCM values to eight digit strings; invalid lengths become missing."""
    ncm = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    ncm = ncm.str.replace(r"\D", "", regex=True)
    ncm = ncm.mask(ncm.eq(""), pd.NA).str.zfill(8)
    return ncm.mask(ncm.str.len() != 8, pd.NA)


def load_target_ncms(csv_path):
    """Load and normalize the exact NCM codes selected by the supermarket analysis."""
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV de NCMs nao encontrado: {csv_path}")
    table = pd.read_csv(csv_path, sep=";", dtype={"numero_ncm": "string"})
    if "numero_ncm" not in table.columns:
        raise KeyError("O CSV precisa conter a coluna 'numero_ncm'.")
    return set(normalize_ncm(table["numero_ncm"]).dropna())

## 1. Carregar e validar a lista de NCMs

A lista e lida como texto para preservar zeros a esquerda. Valores que nao resultem em um codigo de oito digitos sao descartados na normalizacao.

In [3]:
target_ncms = load_target_ncms(NCM_CSV_PATH)
if not target_ncms:
    raise ValueError("Nenhum NCM valido de oito digitos foi encontrado no CSV.")

parquet_paths = sorted(SOURCE_ITEMS_DIR.glob("*.parquet"))
if not parquet_paths:
    raise FileNotFoundError(f"Nenhum Parquet mensal encontrado em {SOURCE_ITEMS_DIR}")

sample = pd.read_parquet(parquet_paths[0])
ncm_col = find_col([column for column in sample.columns if "TIPO" not in str(column).upper()], "NCM/SH")
desc_col = find_col(sample.columns, "PRODUTO/SERV")

display(pd.DataFrame({
    "metrica": ["NCMs validos no CSV", "arquivos mensais de origem", "coluna NCM", "coluna descricao"],
    "valor": [len(target_ncms), len(parquet_paths), ncm_col, desc_col],
}))

,metrica,valor
0,NCMs validos no CSV,241
1,arquivos mensais de origem,12
2,coluna NCM,CÓDIGO NCM/SH
3,coluna descricao,DESCRIÇÃO DO PRODUTO/SERVIÇO


## 2. Conferir cobertura do filtro

A tabela abaixo mostra, por mes, quantos itens possuem descricao e quantos correspondem exatamente a um dos NCMs da lista.

In [4]:
preview_summary = []
for parquet_path in parquet_paths:
    frame = pd.read_parquet(parquet_path, columns=[desc_col, ncm_col])
    items = frame.loc[frame[desc_col].notna()]
    normalized_ncm = normalize_ncm(items[ncm_col])
    preview_summary.append({
        "arquivo": parquet_path.name,
        "itens_com_descricao": len(items),
        "itens_com_ncm_valido": int(normalized_ncm.notna().sum()),
        "itens_correspondentes": int(normalized_ncm.isin(target_ncms).sum()),
    })

display(pd.DataFrame(preview_summary))

,arquivo,itens_com_descricao,itens_com_ncm_valido,itens_correspondentes
0,2025-01.parquet,369683,369683,47206
1,2025-02.parquet,490657,490657,61896
2,2025-03.parquet,434549,434549,68386
3,2025-04.parquet,470126,470126,70061
4,2025-05.parquet,523639,523639,81687
5,2025-06.parquet,517843,517843,71937
6,2025-07.parquet,569088,569088,80576
7,2025-08.parquet,489660,489660,72976
8,2025-09.parquet,554569,554569,79779
9,2025-10.parquet,562373,562373,80502


## 3. Gerar os Parquets filtrados

Esta celula grava uma copia filtrada de cada mes e falha se a pasta de saida ja existir, para evitar sobrescrever resultados anteriores. Para gerar uma nova versao, altere `OUTPUT_DIR` na configuracao para outro nome.

In [5]:
if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"A pasta de saida ja existe: {OUTPUT_DIR}. "
        "Escolha um novo nome em OUTPUT_DIR para preservar os arquivos existentes."
    )

OUTPUT_DIR.mkdir(parents=True)
generation_summary = []

for parquet_path in parquet_paths:
    frame = pd.read_parquet(parquet_path)
    item_frame = frame.loc[frame[desc_col].notna()].copy()
    ncm_values = normalize_ncm(item_frame[ncm_col])
    filtered = item_frame.loc[ncm_values.isin(target_ncms)].copy()
    output_path = OUTPUT_DIR / parquet_path.name
    filtered.to_parquet(output_path, index=False)
    generation_summary.append({
        "arquivo": parquet_path.name,
        "itens_origem": len(item_frame),
        "itens_filtrados": len(filtered),
        "saida": str(output_path.relative_to(PROJECT_ROOT)),
    })

display(pd.DataFrame(generation_summary))

,arquivo,itens_origem,itens_filtrados,saida
0,2025-01.parquet,369683,47206,data\interim\2025_NCMs_filtrados\2025-01.parquet
1,2025-02.parquet,490657,61896,data\interim\2025_NCMs_filtrados\2025-02.parquet
2,2025-03.parquet,434549,68386,data\interim\2025_NCMs_filtrados\2025-03.parquet
3,2025-04.parquet,470126,70061,data\interim\2025_NCMs_filtrados\2025-04.parquet
4,2025-05.parquet,523639,81687,data\interim\2025_NCMs_filtrados\2025-05.parquet
5,2025-06.parquet,517843,71937,data\interim\2025_NCMs_filtrados\2025-06.parquet
6,2025-07.parquet,569088,80576,data\interim\2025_NCMs_filtrados\2025-07.parquet
7,2025-08.parquet,489660,72976,data\interim\2025_NCMs_filtrados\2025-08.parquet
8,2025-09.parquet,554569,79779,data\interim\2025_NCMs_filtrados\2025-09.parquet
9,2025-10.parquet,562373,80502,data\interim\2025_NCMs_filtrados\2025-10.parquet
